In [1]:
import pandas as pd

# The Socio-Economic Impact on Education: Predicting State Maturity Exam Scores using Machine Learning

## 1. Problem Statement and Motivation

**Introduction and Context**
Educational outcomes are often viewed as a reflection of a school's quality, but they are deeply intertwined with the broader socio-economic reality of the region. In Bulgaria, the State Maturity Exams (Държавен зрелостен изпит or DZI) serve as the ultimate benchmark for high school education. However, analyzing these grades in a vacuum ignores the macro-economic factors that influence student success, such as regional income disparities, labor market dynamics, and overall living conditions.

**Problem Formulation**
The core objective of this project is to build a Machine Learning regression model that predicts a school's average DZI performance by bridging the gap between micro-level and macro-level data. Specifically, I will investigate whether combining school-specific characteristics (e.g., school profile, cohort size, and historical performance) with district-level socio-economic indicators (e.g., regional income, youth unemployment, and poverty lines) can create a robust predictive model. 

**Project Significance and Objectives**
This approach transforms a simple grade-prediction task into a deeper socio-economic analysis. The primary goals of this project are:
1. **Data Integration:** To successfully merge high-dimensional macro-economic datasets (Regional Profiles) with granular, micro-level educational data (DZI results).
2. **Feature Engineering:** To extract meaningful variables from raw data, such as categorizing school types and utilizing time-series lagged variables for historical performance.
3. **Predictive Modeling:** To train, test, and evaluate various regression algorithms (e.g., Linear Regression, Random Forests) to predict future DZI outcomes.
4. **Interpretability:** To analyze feature importance and understand which socio-economic or school-specific factors carry the most weight in determining educational success.

## 2. Data Loading and Preparation

In [12]:
years = range(2018, 2027)
dzi_frames = []

data_start_mapping = {
    2020: 2, 
    2021: 2, 
    2023: 3
}

for year in years:
    raw_data = pd.read_csv(f'data/dzi-{year}.csv', header=None)
    
    top_rows = raw_data.iloc[0:4].ffill(axis=1).astype(str)
    
    combined_headers = top_rows.apply(lambda x: ' '.join(x).lower())
    
    col_map = {}
    
    for idx, text in enumerate(combined_headers):
        if 'област' in text and 'училище' not in text:
            col_map[idx] = 'District'
        elif 'населено' in text:
            col_map[idx] = 'Settlement'
        elif 'училище' in text and 'профил' not in text:
            col_map[idx] = 'School_Name'
        elif 'админ' in text or 'неиспуо' in text:
            col_map[idx] = 'School_Code'
            
        # Target Variable Extraction
        elif 'бел' in text:
            if any(w in text for w in ['усп', 'оцен']) and 'BEL_Grade' not in col_map.values():
                col_map[idx] = 'BEL_Grade'
            elif any(w in text for w in ['брой', 'бр', 'явили']) and 'BEL_Students_Count' not in col_map.values():
                col_map[idx] = 'BEL_Students_Count'

    yearly_filtered = raw_data[list(col_map.keys())].copy()
    
    yearly_filtered.columns = list(col_map.values())
    yearly_filtered['Year'] = year
    
    start_idx = data_start_mapping.get(year, 1)
    yearly_filtered = yearly_filtered.iloc[start_idx:].copy()
    
    yearly_filtered = yearly_filtered.dropna(subset=['District'])
    yearly_filtered = yearly_filtered[~yearly_filtered['District'].str.lower().str.contains('област')]
            
    dzi_frames.append(yearly_filtered)

# Consolidate the focused dataset
dzi_full = pd.concat(dzi_frames, ignore_index=True)

# Load Macro-Economic Datasets 
income_data = pd.read_csv('data/E1_25.csv')
labor_data = pd.read_csv('data/E2_25.csv')
edu_data = pd.read_csv('data/S2_25.csv')

In [13]:
dzi_full

,District,Settlement,School_Code,School_Name,BEL_Students_Count,BEL_Grade,Year
0,БЛАГОЕВГРАД,ГР.БАНСКО,102015,Професионална гимназия по електроника и енерге...,38,"4,048",2018
1,БЛАГОЕВГРАД,ГР.БАНСКО,102004,"Професионална лесотехническа гимназия""Никола Й...",39,"3,916",2018
2,БЛАГОЕВГРАД,ГР.БАНСКО,102010,"Гимназия по туризъм ""Алеко Константинов""",18,"3,454",2018
3,БЛАГОЕВГРАД,ГР.БЕЛИЦА,105002,"Професионална гимназия ""Никола Йонков Вапцаров""",17,"3,955",2018
4,БЛАГОЕВГРАД,ГР.БЕЛИЦА,105001,"Средно училище ""Св.Св.Кирил и Методий""",39,"4,336",2018
...,...,...,...,...,...,...,...
8331,ЯМБОЛ,ГР.ЯМБОЛ,2811603,"Професионална гимназия по туризъм ""Алеко Конст...",35,3.53,2026
8332,ЯМБОЛ,ГР.ЯМБОЛ,2811604,"Професионална гимназия по земеделие ""Христо Бо...",10,2.74,2026
8333,ЯМБОЛ,ГР.ЯМБОЛ,2811607,"Гимназия по строителство и архитектура, график...",88,4.4,2026
8334,ЯМБОЛ,ГР.ЯМБОЛ,2811608,"Професионална гимназия по подемна, строителна ...",21,2.64,2026


In [4]:
income_data

,Наименование,"БВП на човек от населението, лв.",Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,"Относителен дял на населението, живеещо под линията на бедността за страната, %",Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54
0,Година,2012.0,2013.0,2014.0,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,...,2021.0,2022.0,2023,2024.0,2019.0,2020.0,2021.0,2022,2023.0,2024.0
1,Благоевград,7533.0,7660.0,7659.0,8013.0,8449.0,9069.0,9989.0,10627.0,11179.0,...,34.1,32.1,27.6,28.6,23.9,25.1,19.1,22.9,21.6,19.5
2,Бургас,9881.0,10101.0,9449.0,10943.0,12174.0,13283.0,13460.0,14483.0,12351.0,...,32.6,38.0,29.7,31.7,20.0,26.5,24.6,22.1,21.0,23.0
3,Варна,11631.0,11543.0,12576.0,13285.0,13756.0,14992.0,16537.0,17545.0,17379.0,...,35.9,33.5,33.1,37.1,18.4,22.9,17.3,14.2,14.6,15.2
4,Велико Търново,7586.0,7974.0,8075.0,8665.0,9121.0,9950.0,11262.0,12078.0,12448.0,...,32.1,30.8,32.8,27.3,25.8,30.8,22.5,20.7,28.9,25.7
5,Видин,5695.0,6016.0,6283.0,6494.0,6712.0,7758.0,8357.0,9267.0,9424.0,...,34.2,37.5,33.4,35.2,35.0,43.3,43.3,39.2,33.7,34.9
6,Враца,9187.0,8277.0,9631.0,9375.0,9922.0,12020.0,15258.0,13578.0,16016.0,...,46.9,44.5,39.0,37.8,38.9,33.2,33.5,31.5,19.2,18.0
7,Габрово,9433.0,9221.0,10053.0,10857.0,11706.0,13041.0,14219.0,15375.0,15855.0,...,27.8,30.8,24.3,25.2,17.8,22.3,12.3,21.9,15.7,16.7
8,Добрич,7591.0,7957.0,8185.0,8470.0,9045.0,9826.0,10047.0,11002.0,11803.0,...,31.6,33.6,41.2,38.3,22.8,25.0,26.0,31.0,32.1,26.8
9,Кърджали,6381.0,6228.0,6219.0,6764.0,7243.0,8006.0,9175.0,10799.0,12783.0,...,31.6,31.6,32.7,30.9,35.2,30.8,25.5,25.9,27.3,33.2


In [5]:
labor_data

,Наименование,Относителен дял на населениeто на възраст между 25 и 64 навършени години с висше образование (%),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 58,Unnamed: 59,Unnamed: 60,Unnamed: 61,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67
0,Година,2012.0,2013.0,2014.0,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023,2024.0
1,Благоевград,17.7,18.0,19.6,19.5,19.4,19.9,20.7,21.8,21.1,...,63.0,62.6,62.1,61.7,61.2,61.1,61.0,59.8,59.5,59.2
2,Бургас,18.6,20.2,18.8,19.3,23.1,24.8,23.6,22.5,24.2,...,61.6,61.4,60.9,60.5,60.2,60.1,60.2,58.9,58.7,58.7
3,Варна,26.0,31.4,33.8,30.6,29.9,32.5,29.5,25.3,24.8,...,62.6,62.4,62.1,61.8,61.7,61.7,62.0,60.3,60.4,60.5
4,Велико Търново,23.5,26.6,27.3,26.9,27.3,22.1,26.8,29.6,29.7,...,60.1,59.8,59.5,59.3,59.1,59.1,59.4,56.4,56.3,56.3
5,Видин,15.7,14.6,17.2,21.5,18.5,18.5,20.1,19.2,18.0,...,54.1,54.0,53.9,53.7,53.7,54.2,54.6,53.6,53.5,53.5
6,Враца,18.2,20.4,22.1,21.1,19.2,19.9,21.5,21.0,22.9,...,58.3,58.1,57.8,57.5,57.3,57.4,57.6,57.3,57.2,57.1
7,Габрово,25.1,24.9,26.3,26.8,25.9,26.5,27.9,29.0,23.1,...,56.0,55.8,55.5,55.3,54.9,55.2,55.4,54.3,54.4,54.3
8,Добрич,18.3,18.4,18.4,19.4,22.1,20.9,20.8,21.1,22.3,...,60.2,60.0,59.6,59.3,59.0,59.2,59.3,56.5,56.4,56.3
9,Кърджали,11.2,15.6,19.8,17.2,15.0,13.8,16.8,18.6,17.6,...,62.3,62.0,61.4,60.8,60.1,59.8,59.6,57.0,56.8,56.5


In [6]:
edu_data

,Наименование,Брой на студентите в колежи и университети на 1000 души,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69,Unnamed: 70,Unnamed: 71,Unnamed: 72,Unnamed: 73
0,Година,2012.0,2013.0,2014.0,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024.0
1,Благоевград,41.3,42.9,41.6,38.5,34.9,32.4,30.8,29.6,29.4,...,88.2,88.8,89.2,88.9,88.3,89.4,89.6,92.6,93.8,95.3
2,Бургас,23.6,23.4,21.7,19.8,17.3,15.9,14.4,14.8,15.1,...,84.0,81.5,80.1,79.3,80.0,78.8,79.0,88.4,88.1,89.0
3,Варна,69.3,67.2,65.2,63.8,56.9,52.9,51.0,50.9,50.1,...,81.2,79.8,78.9,79.2,80.1,80.4,81.2,90.3,90.5,90.5
4,Велико Търново,109.3,107.3,108.3,97.4,89.0,77.4,69.5,67.2,68.1,...,79.6,76.4,75.4,76.6,78.1,79.1,80.6,90.6,92.0,92.8
5,Видин,0.0,0.0,0.0,0.0,0.0,2.3,3.8,5.1,5.5,...,81.0,80.8,80.5,78.2,78.7,76.3,80.7,87.1,88.8,90.6
6,Враца,3.3,3.7,4.4,5.0,6.1,5.6,5.8,6.8,7.1,...,87.9,86.4,86.9,86.1,86.3,86.2,88.2,90.6,92.3,93.8
7,Габрово,48.8,50.7,53.1,57.3,57.0,48.9,46.7,43.8,46.1,...,86.9,85.5,85.2,85.9,85.0,85.1,85.7,89.7,91.3,92.3
8,Добрич,6.1,5.9,4.8,3.9,4.2,4.1,4.4,4.3,3.8,...,75.6,75.6,72.5,75.1,76.6,75.6,75.9,86.9,87.2,89.3
9,Кърджали,6.6,6.7,6.7,6.3,5.9,4.7,4.2,3.9,3.7,...,77.7,74.6,73.7,69.8,65.7,61.1,58.7,83.3,78.1,74.8


### 2.1 Data Cleaning: Addressing Hierarchical Headers

**Problem Identification**
During the initial data exploration of the socio-economic datasets (Income, Labor, Education), we encountered a structural inconsistency common to data converted from Excel. The headers were originally merged cells in the source files, which resulted in a "split-header" format when converted to CSV. Specifically, top-level categories were associated with multiple years, but the CSV export left the columns for the subsequent years empty (represented as `Unnamed` in Pandas).

**Methodology for Resolution**
To preserve the integrity of the data and enable effective merging, we implemented a robust cleaning pipeline:
1. **MultiIndex Loading:** We instructed Pandas to parse the first two rows as a hierarchical header (`header=[0, 1]`), correctly identifying the relationship between categories and specific years.
2. **Forward-Filling Categories:** We utilized a forward-fill (`ffill()`) strategy on the category-level header to propagate the label across the relevant years, ensuring that every column is fully identified by its category and its corresponding year.

In [7]:
def clean_macro_data(file_path):
    # 1. Load the file, keeping the first two rows as the header
    raw_data = pd.read_csv(file_path, header=[0, 1])
    
    # 2. The first column name is currently a tuple like ('Наименование', 'Година')
    # Clean that up to be just 'District'
    raw_data.columns.values[0] = ('District', 'District')
    
    # 3. Forward fill the top-level header (the Category names)
    new_cols = pd.MultiIndex.from_tuples(
        [(c[0] if not 'Unnamed' in c[0] else None, c[1]) for c in raw_data.columns]
    )
    
    # Forward fill the categories
    category_cols = pd.Series(new_cols.get_level_values(0)).ffill()
    year_cols = new_cols.get_level_values(1)
    
    raw_data.columns = pd.MultiIndex.from_arrays([category_cols, year_cols])
    
    return raw_data

# Apply the fix
income_data = clean_macro_data('data/E1_25.csv')
labor_data = clean_macro_data('data/E2_25.csv')
edu_data = clean_macro_data('data/S2_25.csv')

# Verification: Display the first few rows and the column structure
print("Cleaned Data Header Structure:")
display(income_data.head(3))
display(labor_data.head(3))
display(edu_data.head(3))

Cleaned Data Header Structure:


District БВП на човек от населението, лв.                             \
      District                             2012     2013     2014     2015   
0  Благоевград                           7533.0   7660.0   7659.0   8013.0   
1       Бургас                           9881.0  10101.0   9449.0  10943.0   
2        Варна                          11631.0  11543.0  12576.0  13285.0   

                                                ...  \
      2016     2017     2018     2019     2020  ...   
0   8449.0   9069.0   9989.0  10627.0  11179.0  ...   
1  12174.0  13283.0  13460.0  14483.0  12351.0  ...   
2  13756.0  14992.0  16537.0  17545.0  17379.0  ...   

  Коефициент на Джини на подоходно неравенство                    \
                                          2021  2022  2023  2024   
0                                         34.1  32.1  27.6  28.6   
1                                         32.6  38.0  29.7  31.7   
2                                         35.9  33.5  33.1  37.1   

  Относителен дял на населението, живеещо под линията на бедността за страната, %  \
                                                                             2019   
0                                               23.9                                
1                                               20.0                                
2                                               18.4                                

                                 
   2020  2021  2022  2023  2024  
0  25.1  19.1  22.9  21.6  19.5  
1  26.5  24.6  22.1  21.0  23.0  
2  22.9  17.3  14.2  14.6  15.2  

[3 rows x 55 columns]

District  \
      District   
0  Благоевград   
1       Бургас   
2        Варна   

  Относителен дял на населениeто на възраст между 25 и 64 навършени години с висше образование (%)  \
                                                                                              2012   
0                                               17.7                                                 
1                                               18.6                                                 
2                                               26.0                                                 

                                                   ...  \
   2013  2014  2015  2016  2017  2018  2019  2020  ...   
0  18.0  19.6  19.5  19.4  19.9  20.7  21.8  21.1  ...   
1  20.2  18.8  19.3  23.1  24.8  23.6  22.5  24.2  ...   
2  31.4  33.8  30.6  29.9  32.5  29.5  25.3  24.8  ...   

  Дял на населението в трудоспособна възраст,%                                \
                                          2015  2016  2017  2018  2019  2020   
0                                         63.0  62.6  62.1  61.7  61.2  61.1   
1                                         61.6  61.4  60.9  60.5  60.2  60.1   
2                                         62.6  62.4  62.1  61.8  61.7  61.7   

                           
   2021  2022  2023  2024  
0  61.0  59.8  59.5  59.2  
1  60.2  58.9  58.7  58.7  
2  62.0  60.3  60.4  60.5  

[3 rows x 68 columns]

District Брой на студентите в колежи и университети на 1000 души        \
      District                                                    2012  2013   
0  Благоевград                                               41.3       42.9   
1       Бургас                                               23.6       23.4   
2        Варна                                               69.3       67.2   

                                             ...  \
   2014  2015  2016  2017  2018  2019  2020  ...   
0  41.6  38.5  34.9  32.4  30.8  29.6  29.4  ...   
1  21.7  19.8  17.3  15.9  14.4  14.8  15.1  ...   
2  65.2  63.8  56.9  52.9  51.0  50.9  50.1  ...   

  Дял на децата в детските градини (Групов нетен коефициент на записване)  \
                                                                     2015   
0                                               88.2                        
1                                               84.0                        
2                                               81.2                        

                                                         
   2016  2017  2018  2019  2020  2021  2022  2023  2024  
0  88.8  89.2  88.9  88.3  89.4  89.6  92.6  93.8  95.3  
1  81.5  80.1  79.3  80.0  78.8  79.0  88.4  88.1  89.0  
2  79.8  78.9  79.2  80.1  80.4  81.2  90.3  90.5  90.5  

[3 rows x 74 columns]

## 3. Data Integration

### 3.1 Reshaping Macro Data (Wide to Long Format)

**Objective**
Currently, our macro-economic datasets (`income_data`, `labor_data`, and `edu_data`) are in a "wide" format, where separate years exist as distinct columns under each indicator category. However, our main school dataset (`dzi_full`) is organized in a "long" format, where each row represents a specific school's performance in a single year. To merge these datasets seamlessly, we must reshape the macro data so that every row represents a unique combination of a `District` and a single `Year`.

**Methodology**
Instead of using a flat `pd.melt()`, which is structurally difficult to apply to multi-level indexes, we leverage the advanced Pandas `.stack()` method. By stacking the second level of our column hierarchy (the years), we dynamically rotate the time dimensions from columns into rows. This naturally transforms the dataset into a longitudinal layout while preserving the indicator names as top-level features.

In [8]:
def transform_wide_to_long(macro_table):
    district_series = macro_table[('District', 'District')].astype(str).str.strip()
    
    if (district_series == 'България').any():
        bg_index = district_series[district_series == 'България'].index[0]
        filtered_table = macro_table.iloc[:bg_index].copy()
    else:
        filtered_table = macro_table[~district_series.str.contains('Източник', na=False, case=False)].copy()
    
    macro_indexed = filtered_table.set_index(('District', 'District'))
    macro_indexed.index.name = 'District'
    
    long_format_data = macro_indexed.stack(level=1)
    
    long_format_data = long_format_data.reset_index()
    
    long_format_data.columns.values[1] = 'Year'
    
    long_format_data['Year'] = pd.to_numeric(long_format_data['Year'], errors='coerce')
    long_format_data = long_format_data.dropna(subset=['Year'])
    long_format_data['Year'] = long_format_data['Year'].astype(int)
    
    return long_format_data

income_long = transform_wide_to_long(income_data)
labor_long = transform_wide_to_long(labor_data)
edu_long = transform_wide_to_long(edu_data)

/var/folders/75/l9c923tx7lq_kr4c7t01qm5c0000gn/T/ipykernel_24649/409656948.py:13: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  long_format_data = macro_indexed.stack(level=1)
/var/folders/75/l9c923tx7lq_kr4c7t01qm5c0000gn/T/ipykernel_24649/409656948.py:13: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  long_format_data = macro_indexed.stack(level=1)
/var/folders/75/l9c923tx7lq_kr4c7t01qm5c0000gn/T/ipykernel_24649/409656948.py:13: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes 

In [9]:
income_long

,District,Year,"БВП на човек от населението, лв.","Средна годишна брутна заплата на наетите по трудово и служебно правоотношение, лв.","Размер на средна пенсия, лв.",Коефициент на Джини на подоходно неравенство,"Относителен дял на населението, живеещо под линията на бедността за страната, %"
0,Благоевград,2012,7533.0,6271.0,NaN,23.3,NaN
1,Благоевград,2013,7660.0,6566.0,NaN,25.5,NaN
2,Благоевград,2014,7659.0,6818.0,280.0,27.6,NaN
3,Благоевград,2015,8013.0,7181.0,290.0,26.9,NaN
4,Благоевград,2016,8449.0,7658.0,299.0,32.1,NaN
...,...,...,...,...,...,...,...
359,Ямбол,2020,11135.0,13055.0,425.0,37.7,31.4
360,Ямбол,2021,13278.0,14779.0,521.0,34.7,24.0
361,Ямбол,2022,16348,16472,654.0,33.7,20.2
362,Ямбол,2023,17865.0,18355.0,763,34.2,26.7


In [10]:
labor_long

,District,Year,Относителен дял на населениeто на възраст между 25 и 64 навършени години с висше образование (%),Относителен дял на населението на възраст между 25 и 64 навършени години с основно и по-ниско образование (%),"Средногодишен коефициент на безработица, %","Средногодишен коефициент на заетост на населението на 20-64 години, %","Дял на населението в трудоспособна възраст,%"
0,Благоевград,2010,NaN,NaN,NaN,70.4,NaN
1,Благоевград,2011,NaN,NaN,NaN,69.6,NaN
2,Благоевград,2012,17.7,17.5,14.1,70.1,64.3
3,Благоевград,2013,18.0,22.0,14.3,67.8,64.1
4,Благоевград,2014,19.6,23.8,15.0,67.0,63.5
...,...,...,...,...,...,...,...
415,Ямбол,2020,23.5,21.3,7.0,68.8,56.1
416,Ямбол,2021,23.0,20.6,5.5,67.3,56.1
417,Ямбол,2022,19.7,21.8,5.4,73.7,54.2
418,Ямбол,2023,20.0,18.6,5.1,74.8,54.1


In [11]:
edu_long

,District,Year,Брой на студентите в колежи и университети на 1000 души,"Нетен коефициент на записване на населението V-VII, %","Дял на оценките на ДЗИ по български език и литература под среден 3,00, %","Среден успех на ДЗИ по български език и литература, /6","Среден успех на НВО след 7 клас по математика, в точки/100","Индекс на съответствието между ПОО и заетостта, в точки/100",Дял на децата в детските градини (Групов нетен коефициент на записване)
0,Благоевград,2012,41.3,NaN,5.4,4.19,NaN,NaN,84.8
1,Благоевград,2013,42.9,NaN,3.8,4.29,NaN,NaN,87.8
2,Благоевград,2014,41.6,NaN,2.7,4.41,NaN,NaN,89.3
3,Благоевград,2015,38.5,NaN,6.1,4.24,NaN,NaN,88.2
4,Благоевград,2016,34.9,NaN,10.5,4.04,NaN,NaN,88.8
...,...,...,...,...,...,...,...,...,...
387,Ямбол,2021,5.8,89.1,13.3,3.94,35.0,NaN,77.3
388,Ямбол,2022,6.7,90.0,21.0,3.79,30.7,54.6,85.4
389,Ямбол,2023,6.3,89.3,16.2,3.79,30.8,56.5,86.4
390,Ямбол,2024,7.2,88.8,7.1,4.25,40.4,58.9,86.5
